#Setup

In [65]:
!pip3 install wheel --quiet
!pip3 install numpy scipy dataclasses plotly pillow --quiet

In [66]:
# b) download from github to your PC first and upload "indoor-location-competition-20-master.zip" to your own Google Drive.
# everytime you start a Colab session, you mount Google Drive and copy it from there.
from google.colab import drive
drive.mount('/content/drive/') # mount your drive
!cp "/content/drive/My   Drive/AI6128_UrbanComputing/indoor-location-competition-20-master.zip" ./  # copy to the Colab machine
!unzip -q ./indoor-location-competition-20-master.zip # unzip the file
!mv ./indoor-location-competition-20-master/* ./

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
replace indoor-location-competition-20-master/.gitignore? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
mv: cannot overwrite './data': Directory not empty


In [67]:
# install necessary packages
!pip install numpy scipy dataclasses plotly pillow

In [117]:
# this cell contains the content from compute_f.py
# figuring out localization based on IMU data
import numpy as np
import scipy.signal as signal


def split_ts_seq(ts_seq, sep_ts):
    """

    :param ts_seq:
    :param sep_ts:
    :return:
    """
    tss = ts_seq[:, 0].astype(float)
    unique_sep_ts = np.unique(sep_ts)
    ts_seqs = []
    start_index = 0
    for i in range(0, unique_sep_ts.shape[0]):
        end_index = np.searchsorted(tss, unique_sep_ts[i], side='right')
        if start_index == end_index:
            continue
        ts_seqs.append(ts_seq[start_index:end_index, :].copy())
        start_index = end_index

    # tail data
    if start_index < ts_seq.shape[0]:
        ts_seqs.append(ts_seq[start_index:, :].copy())

    return ts_seqs


def correct_trajectory(original_xys, end_xy):
    """

    :param original_xys: numpy ndarray, shape(N, 2)
    :param end_xy: numpy ndarray, shape(1, 2)
    :return:
    """
    corrected_xys = np.zeros((0, 2))

    A = original_xys[0, :]
    B = end_xy
    Bp = original_xys[-1, :]
    angle_BAX = np.arctan2(B[1] - A[1], B[0] - A[0])
    angle_BpAX = np.arctan2(Bp[1] - A[1], Bp[0] - A[0])
    angle_BpAB = angle_BpAX - angle_BAX
    AB = np.sqrt(np.sum((B - A) ** 2))
    ABp = np.sqrt(np.sum((Bp - A) ** 2))

    corrected_xys = np.append(corrected_xys, [A], 0)
    for i in np.arange(1, np.size(original_xys, 0)):
        angle_CpAX = np.arctan2(original_xys[i, 1] - A[1], original_xys[i, 0] - A[0])

        angle_CAX = angle_CpAX - angle_BpAB

        ACp = np.sqrt(np.sum((original_xys[i, :] - A) ** 2))

        AC = ACp * AB / ABp

        delta_C = np.array([AC * np.cos(angle_CAX), AC * np.sin(angle_CAX)])

        C = delta_C + A

        corrected_xys = np.append(corrected_xys, [C], 0)

    return corrected_xys


def correct_positions(rel_positions, reference_positions):
    """

    :param rel_positions:
    :param reference_positions:
    :return:
    """
    rel_positions_list = split_ts_seq(rel_positions, reference_positions[:, 0])
    if len(rel_positions_list) != reference_positions.shape[0] - 1:
        # print(f'Rel positions list size: {len(rel_positions_list)}, ref positions size: {reference_positions.shape[0]}')
        del rel_positions_list[-1]
    assert len(rel_positions_list) == reference_positions.shape[0] - 1

    corrected_positions = np.zeros((0, 3))
    for i, rel_ps in enumerate(rel_positions_list):
        start_position = reference_positions[i]
        end_position = reference_positions[i + 1]
        abs_ps = np.zeros(rel_ps.shape)
        abs_ps[:, 0] = rel_ps[:, 0]
        # abs_ps[:, 1:3] = rel_ps[:, 1:3] + start_position[1:3]
        abs_ps[0, 1:3] = rel_ps[0, 1:3] + start_position[1:3]
        for j in range(1, rel_ps.shape[0]):
            abs_ps[j, 1:3] = abs_ps[j-1, 1:3] + rel_ps[j, 1:3]
        abs_ps = np.insert(abs_ps, 0, start_position, axis=0)
        corrected_xys = correct_trajectory(abs_ps[:, 1:3], end_position[1:3])
        corrected_ps = np.column_stack((abs_ps[:, 0], corrected_xys))
        if i == 0:
            corrected_positions = np.append(corrected_positions, corrected_ps, axis=0)
        else:
            corrected_positions = np.append(corrected_positions, corrected_ps[1:], axis=0)

    corrected_positions = np.array(corrected_positions)

    return corrected_positions


def init_parameters_filter(sample_freq, warmup_data, cut_off_freq=2):
    order = 4
    filter_b, filter_a = signal.butter(order, cut_off_freq / (sample_freq / 2), 'low', False)
    zf = signal.lfilter_zi(filter_b, filter_a)
    _, zf = signal.lfilter(filter_b, filter_a, warmup_data, zi=zf)
    _, filter_zf = signal.lfilter(filter_b, filter_a, warmup_data, zi=zf)

    return filter_b, filter_a, filter_zf


def get_rotation_matrix_from_vector(rotation_vector):
    q1 = rotation_vector[0]
    q2 = rotation_vector[1]
    q3 = rotation_vector[2]

    if rotation_vector.size >= 4:
        q0 = rotation_vector[3]
    else:
        q0 = 1 - q1*q1 - q2*q2 - q3*q3
        if q0 > 0:
            q0 = np.sqrt(q0)
        else:
            q0 = 0

    sq_q1 = 2 * q1 * q1
    sq_q2 = 2 * q2 * q2
    sq_q3 = 2 * q3 * q3
    q1_q2 = 2 * q1 * q2
    q3_q0 = 2 * q3 * q0
    q1_q3 = 2 * q1 * q3
    q2_q0 = 2 * q2 * q0
    q2_q3 = 2 * q2 * q3
    q1_q0 = 2 * q1 * q0

    R = np.zeros((9,))
    if R.size == 9:
        R[0] = 1 - sq_q2 - sq_q3
        R[1] = q1_q2 - q3_q0
        R[2] = q1_q3 + q2_q0

        R[3] = q1_q2 + q3_q0
        R[4] = 1 - sq_q1 - sq_q3
        R[5] = q2_q3 - q1_q0

        R[6] = q1_q3 - q2_q0
        R[7] = q2_q3 + q1_q0
        R[8] = 1 - sq_q1 - sq_q2

        R = np.reshape(R, (3, 3))
    elif R.size == 16:
        R[0] = 1 - sq_q2 - sq_q3
        R[1] = q1_q2 - q3_q0
        R[2] = q1_q3 + q2_q0
        R[3] = 0.0

        R[4] = q1_q2 + q3_q0
        R[5] = 1 - sq_q1 - sq_q3
        R[6] = q2_q3 - q1_q0
        R[7] = 0.0

        R[8] = q1_q3 - q2_q0
        R[9] = q2_q3 + q1_q0
        R[10] = 1 - sq_q1 - sq_q2
        R[11] = 0.0

        R[12] = R[13] = R[14] = 0.0
        R[15] = 1.0

        R = np.reshape(R, (4, 4))

    return R


def get_orientation(R):
    flat_R = R.flatten()
    values = np.zeros((3,))
    if np.size(flat_R) == 9:
        values[0] = np.arctan2(flat_R[1], flat_R[4])
        values[1] = np.arcsin(-flat_R[7])
        values[2] = np.arctan2(-flat_R[6], flat_R[8])
    else:
        values[0] = np.arctan2(flat_R[1], flat_R[5])
        values[1] = np.arcsin(-flat_R[9])
        values[2] = np.arctan2(-flat_R[8], flat_R[10])

    return values


def compute_steps(acce_datas):
    step_timestamps = np.array([])
    step_indexs = np.array([], dtype=int)
    step_acce_max_mins = np.zeros((0, 4))
    sample_freq = 50
    window_size = 22
    low_acce_mag = 0.6
    step_criterion = 1
    interval_threshold = 250

    acce_max = np.zeros((2,))
    acce_min = np.zeros((2,))
    acce_binarys = np.zeros((window_size,), dtype=int)
    acce_mag_pre = 0
    state_flag = 0

    warmup_data = np.ones((window_size,)) * 9.81
    filter_b, filter_a, filter_zf = init_parameters_filter(sample_freq, warmup_data)
    acce_mag_window = np.zeros((window_size, 1))

    # detect steps according to acceleration magnitudes
    for i in np.arange(0, np.size(acce_datas, 0)):
        acce_data = acce_datas[i, :]
        acce_mag = np.sqrt(np.sum(acce_data[1:] ** 2))

        acce_mag_filt, filter_zf = signal.lfilter(filter_b, filter_a, [acce_mag], zi=filter_zf)
        acce_mag_filt = acce_mag_filt[0]

        acce_mag_window = np.append(acce_mag_window, [acce_mag_filt])
        acce_mag_window = np.delete(acce_mag_window, 0)
        mean_gravity = np.mean(acce_mag_window)
        acce_std = np.std(acce_mag_window)
        mag_threshold = np.max([low_acce_mag, 0.4 * acce_std])

        # detect valid peak or valley of acceleration magnitudes
        acce_mag_filt_detrend = acce_mag_filt - mean_gravity
        if acce_mag_filt_detrend > np.max([acce_mag_pre, mag_threshold]):
            # peak
            acce_binarys = np.append(acce_binarys, [1])
            acce_binarys = np.delete(acce_binarys, 0)
        elif acce_mag_filt_detrend < np.min([acce_mag_pre, -mag_threshold]):
            # valley
            acce_binarys = np.append(acce_binarys, [-1])
            acce_binarys = np.delete(acce_binarys, 0)
        else:
            # between peak and valley
            acce_binarys = np.append(acce_binarys, [0])
            acce_binarys = np.delete(acce_binarys, 0)

        if (acce_binarys[-1] == 0) and (acce_binarys[-2] == 1):
            if state_flag == 0:
                acce_max[:] = acce_data[0], acce_mag_filt
                state_flag = 1
            elif (state_flag == 1) and ((acce_data[0] - acce_max[0]) <= interval_threshold) and (
                    acce_mag_filt > acce_max[1]):
                acce_max[:] = acce_data[0], acce_mag_filt
            elif (state_flag == 2) and ((acce_data[0] - acce_max[0]) > interval_threshold):
                acce_max[:] = acce_data[0], acce_mag_filt
                state_flag = 1

        # choose reasonable step criterion and check if there is a valid step
        # save step acceleration data: step_acce_max_mins = [timestamp, max, min, variance]
        step_flag = False
        if step_criterion == 2:
            if (acce_binarys[-1] == -1) and ((acce_binarys[-2] == 1) or (acce_binarys[-2] == 0)):
                step_flag = True
        elif step_criterion == 3:
            if (acce_binarys[-1] == -1) and (acce_binarys[-2] == 0) and (np.sum(acce_binarys[:-2]) > 1):
                step_flag = True
        else:
            if (acce_binarys[-1] == 0) and acce_binarys[-2] == -1:
                if (state_flag == 1) and ((acce_data[0] - acce_min[0]) > interval_threshold):
                    acce_min[:] = acce_data[0], acce_mag_filt
                    state_flag = 2
                    step_flag = True
                elif (state_flag == 2) and ((acce_data[0] - acce_min[0]) <= interval_threshold) and (
                        acce_mag_filt < acce_min[1]):
                    acce_min[:] = acce_data[0], acce_mag_filt
        if step_flag:
            step_timestamps = np.append(step_timestamps, acce_data[0])
            step_indexs = np.append(step_indexs, [i])
            step_acce_max_mins = np.append(step_acce_max_mins,
                                            [[acce_data[0], acce_max[1], acce_min[1], acce_std ** 2]], axis=0)
        acce_mag_pre = acce_mag_filt_detrend

    return step_timestamps, step_indexs, step_acce_max_mins


def compute_stride_length(step_acce_max_mins):
    K = 0.4
    K_max = 0.8
    K_min = 0.4
    para_a0 = 0.21468084
    para_a1 = 0.09154517
    para_a2 = 0.02301998

    stride_lengths = np.zeros((step_acce_max_mins.shape[0], 2))
    k_real = np.zeros((step_acce_max_mins.shape[0], 2))
    step_timeperiod = np.zeros((step_acce_max_mins.shape[0] - 1, ))
    stride_lengths[:, 0] = step_acce_max_mins[:, 0]
    window_size = 2
    step_timeperiod_temp = np.zeros((0, ))

    # calculate every step period - step_timeperiod unit: second
    for i in range(0, step_timeperiod.shape[0]):
        step_timeperiod_data = (step_acce_max_mins[i + 1, 0] - step_acce_max_mins[i, 0]) / 1000
        step_timeperiod_temp = np.append(step_timeperiod_temp, [step_timeperiod_data])
        if step_timeperiod_temp.shape[0] > window_size:
            step_timeperiod_temp = np.delete(step_timeperiod_temp, [0])
        step_timeperiod[i] = np.sum(step_timeperiod_temp) / step_timeperiod_temp.shape[0]

    # calculate parameters by step period and acceleration magnitude variance
    k_real[:, 0] = step_acce_max_mins[:, 0]
    k_real[0, 1] = K
    for i in range(0, step_timeperiod.shape[0]):
        k_real[i + 1, 1] = np.max([(para_a0 + para_a1 / step_timeperiod[i] + para_a2 * step_acce_max_mins[i, 3]), K_min])
        k_real[i + 1, 1] = np.min([k_real[i + 1, 1], K_max]) * (K / K_min)

    # calculate every stride length by parameters and max and min data of acceleration magnitude
    stride_lengths[:, 1] = np.max([(step_acce_max_mins[:, 1] - step_acce_max_mins[:, 2]),
                                   np.ones((step_acce_max_mins.shape[0], ))], axis=0)**(1 / 4) * k_real[:, 1]

    return stride_lengths


def compute_headings(ahrs_datas):
    headings = np.zeros((np.size(ahrs_datas, 0), 2))
    for i in np.arange(0, np.size(ahrs_datas, 0)):
        ahrs_data = ahrs_datas[i, :]
        rot_mat = get_rotation_matrix_from_vector(ahrs_data[1:])
        azimuth, pitch, roll = get_orientation(rot_mat)
        around_z = (-azimuth) % (2 * np.pi)
        headings[i, :] = ahrs_data[0], around_z
    return headings


def compute_step_heading(step_timestamps, headings):
    step_headings = np.zeros((len(step_timestamps), 2))
    step_timestamps_index = 0
    for i in range(0, len(headings)):
        if step_timestamps_index < len(step_timestamps):
            if headings[i, 0] == step_timestamps[step_timestamps_index]:
                step_headings[step_timestamps_index, :] = headings[i, :]
                step_timestamps_index += 1
        else:
            break
    assert step_timestamps_index == len(step_timestamps)

    return step_headings


def compute_rel_positions(stride_lengths, step_headings):
    rel_positions = np.zeros((stride_lengths.shape[0], 3))
    for i in range(0, stride_lengths.shape[0]):
        rel_positions[i, 0] = stride_lengths[i, 0]
        rel_positions[i, 1] = -stride_lengths[i, 1] * np.sin(step_headings[i, 1])
        rel_positions[i, 2] = stride_lengths[i, 1] * np.cos(step_headings[i, 1])

    return rel_positions

#this is the function linking the small ones, the input to this function is the acceleration data details, ahrs is orientation data from smart phone, posi is the ground truth data
# we can try to use other data other than acceleration
def compute_step_positions(acce_datas, ahrs_datas, posi_datas):
    step_timestamps, step_indexs, step_acce_max_mins = compute_steps(acce_datas)
    # compute_headings is we want to know which direction we are facing when walking, so "headings" tells us which direction we are facing
    headings = compute_headings(ahrs_datas)
    # figure out how long each step taken is
    stride_lengths = compute_stride_length(step_acce_max_mins)
    # to combine the first 2 function
    step_headings = compute_step_heading(step_timestamps, headings)
    # figure out the relative position in the map
    rel_positions = compute_rel_positions(stride_lengths, step_headings)
    # find the corrected positions to ignore the drifts and everything
    step_positions = correct_positions(rel_positions, posi_datas)

    return step_positions


In [118]:
# this cell contain the content from io_f.py
# no need modify anything
from dataclasses import dataclass

import numpy as np


@dataclass
class ReadData:
    acce: np.ndarray
    acce_uncali: np.ndarray
    gyro: np.ndarray
    gyro_uncali: np.ndarray
    magn: np.ndarray
    magn_uncali: np.ndarray
    ahrs: np.ndarray
    wifi: np.ndarray
    ibeacon: np.ndarray
    waypoint: np.ndarray


def read_data_file(data_filename):
    acce = []
    acce_uncali = []
    gyro = []
    gyro_uncali = []
    magn = []
    magn_uncali = []
    ahrs = []
    wifi = []
    ibeacon = []
    waypoint = []

    with open(data_filename, 'r', encoding='utf-8') as file:
        lines = file.readlines()

    for line_data in lines:
        line_data = line_data.strip()
        if not line_data or line_data[0] == '#':
            continue

        line_data = line_data.split('\t')

        if line_data[1] == 'TYPE_ACCELEROMETER':
            acce.append([int(line_data[0]), float(line_data[2]), float(line_data[3]), float(line_data[4])])
            continue

        if line_data[1] == 'TYPE_ACCELEROMETER_UNCALIBRATED':
            acce_uncali.append([int(line_data[0]), float(line_data[2]), float(line_data[3]), float(line_data[4])])
            continue

        if line_data[1] == 'TYPE_GYROSCOPE':
            gyro.append([int(line_data[0]), float(line_data[2]), float(line_data[3]), float(line_data[4])])
            continue

        if line_data[1] == 'TYPE_GYROSCOPE_UNCALIBRATED':
            gyro_uncali.append([int(line_data[0]), float(line_data[2]), float(line_data[3]), float(line_data[4])])
            continue

        if line_data[1] == 'TYPE_MAGNETIC_FIELD':
            magn.append([int(line_data[0]), float(line_data[2]), float(line_data[3]), float(line_data[4])])
            continue

        if line_data[1] == 'TYPE_MAGNETIC_FIELD_UNCALIBRATED':
            magn_uncali.append([int(line_data[0]), float(line_data[2]), float(line_data[3]), float(line_data[4])])
            continue

        if line_data[1] == 'TYPE_ROTATION_VECTOR':
            ahrs.append([int(line_data[0]), float(line_data[2]), float(line_data[3]), float(line_data[4])])
            continue

        if line_data[1] == 'TYPE_WIFI':
            sys_ts = line_data[0]
            ssid = line_data[2]
            bssid = line_data[3]
            rssi = line_data[4]
            lastseen_ts = line_data[6]
            wifi_data = [sys_ts, ssid, bssid, rssi, lastseen_ts]
            wifi.append(wifi_data)
            continue

        if line_data[1] == 'TYPE_BEACON':
            ts = line_data[0]
            uuid = line_data[2]
            major = line_data[3]
            minor = line_data[4]
            rssi = line_data[6]
            ibeacon_data = [ts, '_'.join([uuid, major, minor]), rssi]
            ibeacon.append(ibeacon_data)
            continue

        if line_data[1] == 'TYPE_WAYPOINT':
            waypoint.append([int(line_data[0]), float(line_data[2]), float(line_data[3])])

    acce = np.array(acce)
    acce_uncali = np.array(acce_uncali)
    gyro = np.array(gyro)
    gyro_uncali = np.array(gyro_uncali)
    magn = np.array(magn)
    magn_uncali = np.array(magn_uncali)
    ahrs = np.array(ahrs)
    wifi = np.array(wifi)
    ibeacon = np.array(ibeacon)
    waypoint = np.array(waypoint)

    return ReadData(acce, acce_uncali, gyro, gyro_uncali, magn, magn_uncali, ahrs, wifi, ibeacon, waypoint)


In [119]:
from pathlib import Path

floor_data_dir = './data/site1/F1'
path_data_dir = floor_data_dir + '/path_data_files'
floor_plan_filename = floor_data_dir + '/floor_image.png'
floor_info_filename = floor_data_dir + '/floor_info.json'

save_dir = './output/site1/F1'
path_image_save_dir = save_dir + '/path_images'
step_position_image_save_dir = save_dir
magn_image_save_dir = save_dir
wifi_image_save_dir = save_dir + '/wifi_images'
ibeacon_image_save_dir = save_dir + '/ibeacon_images'
wifi_count_image_save_dir = save_dir
path_filenames = sorted(Path(path_data_dir).resolve().glob("*.txt"))


#Preprocessing

In [120]:
def stitch_mag_pos_data(mag_data, pos_data):
  mag_data = np.asarray(mag_data)
  pos_data = np.asarray(pos_data)
  mag_filtered = []
  for row in pos_data:
    idx = np.argmin(np.abs(mag_data[:,0]-row[0]))
    mag = mag_data[idx,1:4]
    mag_filtered.append([row[0],row[1],row[2],*mag])
  return np.array(mag_filtered)


In [121]:
def preprocess_mag_data(magn_data,acc_data,ahrs_data,waypoint_data):
  magn_data = sorted(magn_data, key=lambda x: x[0,0])
  magn_data = [mag for grp in magn_data for mag in grp]
  waypoint_data = sorted(waypoint_data, key=lambda x: x[0,0])
  waypoint_data = [pnt for grp in waypoint_data for pnt in grp]
  acc_data = sorted(acc_data, key=lambda x: x[0,0])
  acc_data = [pnt for grp in acc_data for pnt in grp]
  ahrs_data = sorted(ahrs_data, key=lambda x: x[0,0])
  ahrs_data = [pnt for grp in ahrs_data for pnt in grp]
  magn_data = np.array(magn_data)
  waypoint_data = np.array(waypoint_data)
  ahrs_data = np.array(ahrs_data)
  acc_data = np.array(acc_data)
  pos_data = compute_step_positions(acc_data, ahrs_data, waypoint_data)

  mag_pos = stitch_mag_pos_data(magn_data, pos_data)
  return mag_pos



In [122]:
def nearest_sensor_reading(sensor_data, timestamps):
  sensor_data = np.asarray(sensor_data)
  timestamps = np.asarray(timestamps)

  ts = sensor_data[:,0]
  idx = np.searchsorted(ts, timestamps)
  idx = np.clip(idx, 1, ts.size-1)
  left = idx-1
  right = idx
  choose_right = np.abs(ts[right]-timestamps)<np.abs(ts[left]-timestamps)
  nearest = np.where(choose_right,right, left)
  return sensor_data[nearest,1:]


In [123]:
magn_data = []
waypoint_data = []
acc_data = []
ahrs_data = []
gyro_data = []
mag_pos_full = []
processed_traj = []
for pth in path_filenames[:]:
  data = read_data_file(pth)
  magn_data.append(data.magn[:])
  waypoint_data.append(data.waypoint[:])
  acc_data.append(data.acce[:])
  ahrs_data.append(data.ahrs[:])
  gyro_data.append(data.gyro[:])
  processed= preprocess_mag_data([data.magn[:]],[data.acce[:]],[data.ahrs[:]],[data.waypoint[:]])
  mag_pos_full.append(processed)
  processed_traj.append(processed)
mag_pos_full = [mag for grp in mag_pos_full for mag in grp]

In [141]:
seq_len = 20
imu_X = []
mag_X = []
y = []
timestamps = []
trajs = []

for traj_id, processed in enumerate(processed_traj):
  ts = processed[:,0]
  acc_vals = nearest_sensor_reading(acc_data[traj_id],ts)
  gyro_vals = nearest_sensor_reading(gyro_data[traj_id],ts)
  ahrs_vals = nearest_sensor_reading(ahrs_data[traj_id],ts)
  combined = np.column_stack([processed,acc_vals,gyro_vals,ahrs_vals])
  for i in range(seq_len-1,len(combined)):
    imu_X.append(combined[i-seq_len+1:i+1,6:].reshape(-1))
    mag_X.append(combined[i-seq_len+1:i+1,3:6].reshape(-1))
    y.append(combined[i,1:3])
    timestamps.append(combined[i,0])
    trajs.append(traj_id)

X_mag = np.asarray(mag_X,dtype = float)
X_imu = np.asarray(imu_X, dtype = float)
y = np.asarray(y,dtype = float)
timestamps = np.asarray(timestamps, dtype = float)
trajs = np.asarray(trajs,dtype = int)
print(X_mag.shape)
print(X_imu.shape)
print(y.shape)
print(timestamps.shape)
print(len(np.unique(trajs)))


(6243, 60)
(6243, 180)
(6243, 2)
(6243,)
113


In [142]:
waypts = [w for traj in waypoint_data for w in traj]
waypts = np.array(waypts,dtype = float)
test_mask = np.zeros(len(y),dtype = bool)

for wp in waypts[:,1:3]:
  test_mask |= np.all(np.isclose(y,wp,atol=1e-6),axis=1)
train_mask =~test_mask

X_mag_train = X_mag[train_mask]
X_imu_train = X_imu[train_mask]
y_train = y[train_mask]
timestamps_train = timestamps[train_mask]
trajs_train = trajs[train_mask]

X_mag_test = X_mag[test_mask]
X_imu_test = X_imu[test_mask]
y_test = y[test_mask]
timestamps_test = timestamps[test_mask]
trajs_test = trajs[test_mask]

print("Train samples",X_mag_train.shape)
print("Test samples",X_mag_test.shape)


Train samples (5578, 60)
Test samples (665, 60)


In [143]:
!pip install scikit-learn

In [144]:
from sklearn.preprocessing import StandardScaler
mag_scaler = StandardScaler()
imu_scaler = StandardScaler()

X_mag_train_s = mag_scaler.fit_transform(X_mag_train)
X_imu_train_s = imu_scaler.fit_transform(X_imu_train)
X_mag_test_s  = mag_scaler.transform(X_mag_test)
X_imu_test_s  = imu_scaler.transform(X_imu_test)


In [145]:
from sklearn.neighbors import NearestNeighbors
K = 20
knn = NearestNeighbors(n_neighbors=K, metric = "euclidean")
knn.fit(X_mag_train_s,y_train)

mag_dist, candidates_idx = knn.kneighbors(X_mag_test_s)
print("candidates shape",candidates_idx.shape)

candidates shape (665, 20)


In [146]:
def viterbi_ranking(beta=0.25,gamma=0.0225,K=20):
  prediction = np.zeros_like(y_test)

  for traj_id in np.unique(trajs_test):
    indices = np.where(trajs_test == traj_id)[0]
    indices = indices[np.argsort(timestamps_test[indices])]
    n = len(indices)
    if(n==0):
      continue
    candidates = np.zeros((n,K,2),dtype = float)
    emissions =  np.zeros((n,K),dtype = float)

    for j,idx in enumerate(indices):
      cand_idx = candidates_idx[idx,:K]
      candidates[j]=y_train[cand_idx]
      imu_dist = np.linalg.norm(X_imu_test_s[idx]-X_imu_train_s[cand_idx],axis=1)
      emissions[j]=mag_dist[idx,:K]+beta*imu_dist

    dp = np.full((n,K),np.inf)
    backptr = np.zeros((n,K),dtype =np.int32)
    dp[0]=emissions[0]
    for t in range(1,n):
      transitions = np.linalg.norm(candidates[t-1][:,None,:]-candidates[t][None,:,:],axis=2)
      for j in range(K):
        costs = dp[t-1]+gamma*transitions[:,j]
        best_prev = np.argmin(costs)
        dp[t,j] = emissions[t,j]+costs[best_prev]
        backptr[t,j] = best_prev
    path = np.zeros(n,dtype=np.int32)
    path[-1]=np.argmin(dp[-1])
    for t in range(n-1,0,-1):
      path[t-1]=backptr[t,path[t]]
    for t,idx in enumerate(indices):
      prediction[idx]=candidates[t,path[t]]
  return prediction


In [147]:
pred_final = viterbi_ranking()
errors = np.linalg.norm(pred_final-y_test,axis=1)
print("Mean error",np.mean(errors))
print("Median error",np.median(errors))

Mean error 1.494354995069876
Median error 0.5905694371097185
